In [1]:
from rdkit import Chem
import pandas as pd
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from rdkit.DataStructs import FingerprintSimilarity

In [2]:
import pandas as pd
import os
import time

from rdkit import Chem
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs


# ---------------------------------------
# 1. File paths
# ---------------------------------------

reference_file = r"D:\DESPU\SEM 3\CAD Practical\Mini Project\BACE1_Enamine_reference_compounds.csv"

enamine_file = r"D:\DESPU\SEM 3\CAD Practical\Mini Project\Enamine_SMILES.csv"

output_file = r"D:\DESPU\SEM 3\CAD Practical\Mini Project\Enamine_similarity_hits.csv"


# ---------------------------------------
# 2. Read the 5 reference compounds
# ---------------------------------------

sm1 = pd.read_csv(reference_file)

# Column containing reference SMILES
reference_smiles = sm1.iloc[:, 2].dropna().tolist()

print("Number of reference compounds:", len(reference_smiles))


# ---------------------------------------
# 3. Generate reference fingerprints ONCE
# ---------------------------------------

reference_fps = []

for smiles in reference_smiles:
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        fp = MACCSkeys.GenMACCSKeys(mol)
        reference_fps.append((smiles, fp))


print("Valid reference compounds:", len(reference_fps))


# ---------------------------------------
# 4. Remove previous output file
# ---------------------------------------

if os.path.exists(output_file):
    os.remove(output_file)


# ---------------------------------------
# 5. Process Enamine CSV in chunks
# ---------------------------------------

chunksize = 100000

total_processed = 0
total_hits = 0

start_time = time.time()

for chunk_number, df in enumerate(
    pd.read_csv(enamine_file, chunksize=chunksize),
    start=1
):

    # Your Enamine SMILES are in column 1
    smiles_list = df.iloc[:, 1].dropna().tolist()

    fingerprints = []
    valid_smiles = []

    # Generate each Enamine fingerprint ONLY ONCE
    for smiles in smiles_list:

        mol = Chem.MolFromSmiles(smiles)

        if mol is not None:

            fp = MACCSkeys.GenMACCSKeys(mol)

            fingerprints.append(fp)
            valid_smiles.append(smiles)


    # ---------------------------------------
    # Compare against all reference compounds
    # ---------------------------------------

    chunk_results = []

    for reference_smiles_value, reference_fp in reference_fps:

        similarities = DataStructs.BulkTanimotoSimilarity(
            reference_fp,
            fingerprints
        )

        for i, similarity in enumerate(similarities):

            if similarity > 0.90:

                chunk_results.append({
                    "Reference_SMILE": reference_smiles_value,
                    "Enamine_SMILE": valid_smiles[i],
                    "Tanimoto_Similarity": similarity
                })


    # ---------------------------------------
    # Save hits immediately
    # ---------------------------------------

    if chunk_results:

        result_df = pd.DataFrame(chunk_results)

        result_df.to_csv(
            output_file,
            mode="a",
            header=not os.path.exists(output_file),
            index=False
        )

        total_hits += len(result_df)


    # ---------------------------------------
    # Progress information
    # ---------------------------------------

    total_processed += len(smiles_list)

    elapsed = time.time() - start_time

    print(
        f"Chunk {chunk_number} completed | "
        f"Processed: {total_processed:,} compounds | "
        f">90% hits: {total_hits:,} | "
        f"Time: {elapsed/60:.1f} min"
    )


print("\n---------------------------------------")
print("Similarity search completed!")
print("---------------------------------------")
print("Total compounds processed:", total_processed)
print("Total >90% hits:", total_hits)
print("Results saved to:")
print(output_file)

Number of reference compounds: 5
Valid reference compounds: 5
Chunk 1 completed | Processed: 100,000 compounds | >90% hits: 0 | Time: 0.8 min
Chunk 2 completed | Processed: 200,000 compounds | >90% hits: 0 | Time: 2.0 min
Chunk 3 completed | Processed: 300,000 compounds | >90% hits: 0 | Time: 3.4 min
Chunk 4 completed | Processed: 400,000 compounds | >90% hits: 0 | Time: 4.9 min
Chunk 5 completed | Processed: 500,000 compounds | >90% hits: 0 | Time: 6.3 min
Chunk 6 completed | Processed: 600,000 compounds | >90% hits: 0 | Time: 7.8 min
Chunk 7 completed | Processed: 700,000 compounds | >90% hits: 0 | Time: 9.3 min
Chunk 8 completed | Processed: 800,000 compounds | >90% hits: 0 | Time: 10.9 min
Chunk 9 completed | Processed: 900,000 compounds | >90% hits: 0 | Time: 12.4 min
Chunk 10 completed | Processed: 1,000,000 compounds | >90% hits: 0 | Time: 13.9 min
Chunk 11 completed | Processed: 1,100,000 compounds | >90% hits: 0 | Time: 15.5 min
Chunk 12 completed | Processed: 1,200,000 compou

KeyboardInterrupt: 